In [1]:
import pandas as pd
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime
from datetime import timedelta
import glob
import geopandas as gpd
import geobr

In [2]:
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams['axes.grid'] = False
plt.rcParams['font.size'] = '14'
plt.rcParams["font.family"] = "Times New Roman"

In [6]:
# variables
threshold_pr_max = 450 #[mm]
threshold_pr_min = 0 #[mm]
file_path = './1 - Organized data gauge/BRAZIL/BRAZIL_DAILY_1961_2024.h5'
cleaned_path = './1 - Organized data gauge/BRAZIL/DATASETS/BRAZIL_DAILY_1961_2024_CLEANED.h5'

# Data processing

In [7]:
df_data = pd.read_hdf(file_path, key = 'table_data', encoding = 'utf-8')
df_data

,gauge_code,datetime,rain_mm
0,110018901A,2021-01-01,6.30
1,110018901A,2021-01-02,1.79
2,110018901A,2021-01-03,0.00
3,110018901A,2021-01-04,16.15
4,110018901A,2021-01-05,1.58
...,...,...,...
2908882,88690050,2023-12-27,0.00
2908883,88690050,2023-12-28,0.00
2908884,88690050,2023-12-29,2.00
2908885,88690050,2023-12-30,0.00


In [8]:
df_info = pd.read_hdf(file_path, key = 'table_info', encoding = 'utf-8')
df_info = df_info[df_info['gauge_code'].isin(df_data['gauge_code'].unique())]
df_info

,gauge_code,state,city,name_station,lat,long,responsible,source,state_abbreviation
0,00047000,PARÁ,SALINÓPOLIS,SALINÓPOLIS,-0.650000,-47.550000,INMET,HIDROWEB,PA
1,00047002,PARÁ,SALINÓPOLIS,SALINÓPOLIS,-0.623100,-47.353600,ANA,HIDROWEB,PA
2,00047003,PARÁ,CURUÇA,CURUÇA,-0.737500,-47.853600,ANA,HIDROWEB,PA
3,00047004,PARÁ,PRIMAVERA,PRIMAVERA,-0.929400,-47.099400,ANA,HIDROWEB,PA
4,00047005,PARÁ,MARAPANIM,MARUDA,-0.633600,-47.658300,ANA,HIDROWEB,PA
...,...,...,...,...,...,...,...,...,...
18977,S713,MATO GROSSO DO SUL,NOVA ANDRADINA,NOVA ANDRADINA | S713,-22.078611,-53.465833,INMET,INMET,MS
18978,S714,MATO GROSSO DO SUL,PEDRO GOMES,PEDRO GOMES | S714,-18.072778,-54.548889,INMET,INMET,MS
18979,S715,MATO GROSSO DO SUL,RIBAS DO RIO PARDO,RIBAS DO RIO PARDO | S715,-20.466694,-53.763028,INMET,INMET,MS
18980,S716,MATO GROSSO DO SUL,SANTA RITA DO PARDO,SANTA RITA DO PARDO | S716,-21.305889,-52.820375,INMET,INMET,MS


In [9]:
gauge_counts = df_info['gauge_code'].value_counts() # Count occurrences of each gauge_code
print(gauge_counts)

gauge_code
00047000      1
251600301C    1
251700101C    1
251650802U    1
251620102A    1
             ..
02040006      1
02040007      1
02040008      1
02040009      1
S717          1
Name: count, Length: 18370, dtype: int64


In [10]:
# df_complete_info = pd.merge(df_data, df_info, on='gauge_code', how = 'inner').sort_values('lat', ascending = True)
# # .dropna(how='any')
# df_complete_info

In [11]:
df_data_filtered = df_data[(df_data['rain_mm'] >= threshold_pr_min) & (df_data['rain_mm'] <= threshold_pr_max)]
df_data_filtered = df_data_filtered[df_data_filtered['gauge_code'].isin(df_info['gauge_code'].unique().tolist())]
df_data_filtered['gauge_code'].unique().tolist()

['110018901A',
 '110020501A',
 '110020502A',
 '120010401A',
 '120032801A',
 '120040101A',
 '120040101H',
 '120040102A',
 '120070801A',
 '130002901A',
 '130006001A',
 '130008601A',
 '130010201A',
 '130014401A',
 '130020101A',
 '130030002A',
 '130050801A',
 '130060701A',
 '130063101A',
 '130063102A',
 '130068001A',
 '130080501A',
 '130090401A',
 '130110002A',
 '130115901A',
 '130150601A',
 '130160501A',
 '130165401A',
 '130170401A',
 '130170402A',
 '130185201A',
 '130185202A',
 '130190203A',
 '130190205A',
 '130200901A',
 '130220701A',
 '130240501A',
 '130240502A',
 '130250405A',
 '130250406A',
 '130260301H',
 '130260302H',
 '130260303H',
 '130260310A',
 '130260326A',
 '130260327A',
 '130260328A',
 '130260329A',
 '130260330A',
 '130260331A',
 '130260332A',
 '130260333A',
 '130260334A',
 '130260335A',
 '130260336A',
 '130260337A',
 '130290001A',
 '130310601A',
 '130330401A',
 '130340301A',
 '130340302A',
 '130340303A',
 '130353601A',
 '130353602A',
 '130356901A',
 '130356902A',
 '13037000

In [12]:
print(len(df_data_filtered), len(df_data_filtered['gauge_code'].unique()), len(df_info), len(df_info['gauge_code'].unique()))

123595839 18370 18370 18370


In [13]:
df_info_filtered = df_info[df_info['gauge_code'].isin(df_data_filtered['gauge_code'].unique())]
# df_info_filtered['gauge_code'].unique().tolist()

print(len(df_info) - len(df_info_filtered), 'gauges were removed due to the precipitation threshold.')

0 gauges were removed due to the precipitation threshold.


In [14]:
print(len(df_data)- len(df_data_filtered), 'data points were removed due to the precipitation threshold.')

21686 data points were removed due to the precipitation threshold.


In [15]:
print((len(df_info) - len(df_info_filtered)), 'gauges were removed due to the precipitation threshold.')

0 gauges were removed due to the precipitation threshold.


In [16]:
print((len(df_data) - len(df_data_filtered))/len(df_data)*100,"%")

0.01754282008153779 %


In [17]:
print((len(df_info) - len(df_info_filtered))/len(df_info)*100,"%")

0.0 %


In [18]:
df_data_filtered.to_hdf(cleaned_path, key='table_data', mode='r+', complevel=9, format = 'table', append = False, complib='blosc:lz4', encoding='utf-8')
df_info_filtered.to_hdf(cleaned_path, key='table_info', mode='r+', complevel=9, format = 'table', append = False, complib='blosc:lz4', encoding='utf-8')

c:\ProgramData\anaconda3\Lib\site-packages\tables\leaf.py:409: RuntimeWarning: overflow encountered in scalar multiply
  expected_mb = (expectedrows * rowsize) // MB


In [19]:
with pd.HDFStore(cleaned_path) as store:
        keys = store.keys()
        print(f"Current tables in {cleaned_path}:")
        for key in keys:
            print(f"  - {key.lstrip('/')}")

Current tables in ./1 - Organized data gauge/BRAZIL/DATASETS/BRAZIL_DAILY_1961_2024_CLEANED.h5:
  - table_data
  - table_data_outlier_filtered
  - table_info
  - table_outlier
  - table_outlier_filter_1
  - table_outlier_filter_1_export
  - table_outlier_filter_2_export
  - table_preclassif


In [20]:
df_info_filtered = pd.read_hdf(cleaned_path, key='table_info', encoding='utf-8')
df_info_filtered

,gauge_code,state,city,name_station,lat,long,responsible,source,state_abbreviation
0,00047000,PARÁ,SALINÓPOLIS,SALINÓPOLIS,-0.650000,-47.550000,INMET,HIDROWEB,PA
1,00047002,PARÁ,SALINÓPOLIS,SALINÓPOLIS,-0.623100,-47.353600,ANA,HIDROWEB,PA
2,00047003,PARÁ,CURUÇA,CURUÇA,-0.737500,-47.853600,ANA,HIDROWEB,PA
3,00047004,PARÁ,PRIMAVERA,PRIMAVERA,-0.929400,-47.099400,ANA,HIDROWEB,PA
4,00047005,PARÁ,MARAPANIM,MARUDA,-0.633600,-47.658300,ANA,HIDROWEB,PA
...,...,...,...,...,...,...,...,...,...
18977,S713,MATO GROSSO DO SUL,NOVA ANDRADINA,NOVA ANDRADINA | S713,-22.078611,-53.465833,INMET,INMET,MS
18978,S714,MATO GROSSO DO SUL,PEDRO GOMES,PEDRO GOMES | S714,-18.072778,-54.548889,INMET,INMET,MS
18979,S715,MATO GROSSO DO SUL,RIBAS DO RIO PARDO,RIBAS DO RIO PARDO | S715,-20.466694,-53.763028,INMET,INMET,MS
18980,S716,MATO GROSSO DO SUL,SANTA RITA DO PARDO,SANTA RITA DO PARDO | S716,-21.305889,-52.820375,INMET,INMET,MS
